# A/B Testing - User Level Dataset Fix

This notebook corrects the dataset to ensure each user belongs to only one experiment group and calculates conversion at user level.

In [1]:
import pandas as pd

events = pd.read_csv('data/events.csv')
transactions = pd.read_csv('data/transactions.csv')

## Load Data

In [3]:
import pandas as pd

events = pd.read_csv('data/events.csv')
transactions = pd.read_csv('data/transactions.csv')

## Conversion flag

In [4]:
transactions['converted'] = 1

## Conversion per user

In [5]:
df_conv = transactions.groupby('customer_id')['converted'].max().reset_index()

## Experiment group fix

In [6]:
df_group = events[['customer_id', 'experiment_group']].drop_duplicates()

df_group = df_group.groupby('customer_id')['experiment_group'].first().reset_index()

## Merge

In [8]:
df_user = df_group.merge(df_conv, on='customer_id', how='left')
df_user['converted'] = df_user['converted'].fillna(0)

## Validation

In [9]:
df_user.head()

,customer_id,experiment_group,converted
0,1,Control,0.0
1,2,Control,1.0
2,3,Control,0.0
3,4,Control,0.0
4,5,Control,0.0


In [10]:
df_user['customer_id'].nunique() == len(df_user)

True

In [11]:
df_user['experiment_group'].value_counts()

Control      60015
Variant_A    20067
Variant_B    19918
Name: experiment_group, dtype: int64

In [12]:
df_user['converted'].value_counts()

1.0    64035
0.0    35965
Name: converted, dtype: int64

In [13]:
df_user.to_csv('ab_test_result.csv', index=False)